# 23 · Randstorm positive control: local placebo pool and the 10-10 notification date

Inputs: `key_pubform`, `panel_rs2` (21); `key_flags`, `exchange_keys`, `custodial_keys`, `large_movers` (04, 04b, 04c)

Outputs: `out23/`, zipped as `out23.zip`: `est_23_pc.csv`, `est_23_pools.csv`, `est_23_calendar_uc.csv`, `est_23_notice_path.csv`, `fig_23_calendar_uc.png`, `fig_23_notice_path.png`, `build_report_23.json`

In 21, Randstorm (2023-11) and FTX (2022-11) precede the placebo pool (from 2024-06) and use the pool mean as baseline. Because the treated–control difference drifts over calendar time, this notebook builds a placebo pool for 2022-02 to 2024-05 with the rules of 03. It also adds 2023-10-10 as an event: Unciphered's announcement of that day states that potentially affected users were notified by e-mail and text message through partner vendors from that day on; technical details followed on 11-14.

- Main specification: local pool `pre`, h = 90 days, `RS2`, main window. Robustness: `pre_src`, `pre_sym30`, h = 60/120, combined pool, wide window; `old_main` (original pool + pool mean) reproduces 21.
- 10-10: estimated at +7 and +30 (window 10-10 to 11-09, no overlap with the 11-14 window), plus a descriptive path from 10-10 to 12-14.

In [ ]:
# Configuration
PROJECT_ID = "your-gcp-project"
DATASET    = "qbtc_raw"
LOCATION   = "US"

AUTO_RUN_USD      = 5.0
MAX_USD_PER_QUERY = 10.0
MAX_USD_TOTAL     = 100.0
PRICE_PER_TIB_USD = 6.25
CONFIRM           = False
FORCE_REBUILD     = False

FREEZE_DATE        = "2026-09-22"
RISK_MIN_BTC       = 0.01        # risk set: minimum balance at t-1
RISK_LOOKBACK_DAYS = 365         # RS2: incoming transfer within the last 12 months (same as the placebo dates)
SERVICE_MAX_RECV   = 1000        # any address with more lifetime receipts -> service/custodian (as in 04)
DORMANT_DAYS       = 30          # y1 strict: unspent for the next 30 days (as in 04)
POST_MAX           = 30          # relative days 0...+30
NOTICE_POST_MAX    = 65          # the 10-10 notification event is followed to +65 (= 30 days after 11-14), descriptive only
LM_MIN_BTC         = 1000        # large transfers: daily y3 >= 1,000 BTC (as in 04c)
COHORT = {"main": ("2012-04-01", "2014-03-31"), "wide": ("2012-04-01", "2015-02-28")}   # as in 21

PRE_FROM, PRE_TO    = "2022-02-01", "2024-06-03"   # range of the local pool; the original main pool starts 2024-06-04
PLACEBO_FROM_03     = "2024-06-04"                 # start of the 03 pool (regression check)
PRE_EXCL, POST_EXCL = 30, 14                       # as in 03: [s - 30, s + 14]
SYM_EXCL            = 30                           # robustness: symmetric +/-30 days
SPIKE_SD            = 2.0                          # as in 03: max attn_pc1 within +/-3 days

H_MAIN, BANDWIDTHS = 90, [60, 90, 120]
REPORT_DAYS = (7, 30)
OUTS = ["y1_strict", "y1_strict_held", "y3", "y0"]
HELD = {"y1_strict_held": "y1_strict"}
STD_TYPES = "('pubkey','pubkeyhash','witness_v0_keyhash','scripthash','witness_v0_scripthash','witness_v1_taproot')"
INPUTS = "bigquery-public-data.crypto_bitcoin.inputs"
OUT_DIR = "out23"

# CEM bins (as in 05; first-seen period and script family are fixed within the cohort)
BAL_EDGES_BTC = [1, 10, 100];     BAL_LABELS   = ["0.01-1", "1-10", "10-100", "100+"]
AGE_EDGES_D   = [365, 730, 1826]; AGE_LABELS   = ["<1y", "1-2y", "2-5y", "5y+"]
REC_EDGES_D   = [30, 180, 365];   REC_LABELS   = ["<30d", "30-180d", "180-365d", "365d+"]
NUTXO_EDGES   = [3, 11];          NUTXO_LABELS = ["1-2", "3-10", "11+"]

# results of 21 (main window, RS2, key-weighted, +30, original main pool, h = 90), for the reproduction check
REF_21 = {("randstorm", "c_y1_strict"): (0.0236255607294543, 0.0235279128448018),
          ("randstorm", "c_h50"):       (0.0234176965677286, 0.0235273245917776),
          ("ftx", "c_y1_strict"):       (-0.0032359956204543, -0.0033336435051068)}   # (theta, beta)

try:
    from google.colab import auth
    auth.authenticate_user()
except ImportError:
    pass

from google.cloud import bigquery
import json, os, re, time, zipfile, datetime as dt
import numpy as np
import pandas as pd
import matplotlib
import matplotlib.pyplot as plt

client = bigquery.Client(project=PROJECT_ID, location=LOCATION)
DS = f"{PROJECT_ID}.{DATASET}"
os.makedirs(OUT_DIR, exist_ok=True)

LEDGER_PATH, JOBS_PATH = "build_ledger.json", "jobs.json"
LEDGER = json.load(open(LEDGER_PATH)) if os.path.exists(LEDGER_PATH) else {"jobs": [], "usd_total": 0.0}
JOBS   = json.load(open(JOBS_PATH))   if os.path.exists(JOBS_PATH)   else []

def usd(gb): return gb / 1024 * PRICE_PER_TIB_USD

def dry_run(sql):
    job = client.query(sql, job_config=bigquery.QueryJobConfig(dry_run=True, use_query_cache=False))
    gb = job.total_bytes_processed / 2**30
    return gb, usd(gb)

def table_rows(name):
    try:
        return client.get_table(f"{DS}.{name}").num_rows
    except Exception:
        return None

def table_info(name):
    t = client.get_table(f"{DS}.{name}")
    return {"table": name, "rows": t.num_rows, "GB": round(t.num_bytes / 2**30, 2)}

def build(name, sql, force=False):
    assert not re.search(r"CLUSTER BY[^\n]*\b(key_id|txid|spending_txid)\b", sql), f"[{name}] cannot cluster on a BYTES column"
    n = table_rows(name)
    if n is not None and not (FORCE_REBUILD or force):
        print(f"[{name}] exists, {n:,} rows; skipped (rebuild with build(..., force=True))")
        return True
    gb, cost = dry_run(sql)
    if cost > MAX_USD_PER_QUERY:
        raise RuntimeError(f"[{name}] estimated ${cost:.2f}, above the per-query limit ${MAX_USD_PER_QUERY}")
    if LEDGER["usd_total"] + cost > MAX_USD_TOTAL:
        raise RuntimeError(f"[{name}] total would exceed ${MAX_USD_TOTAL}")
    if cost > AUTO_RUN_USD and not CONFIRM:
        raise RuntimeError(f"[{name}] estimated {gb:,.0f} GB ~ ${cost:.2f}, above the auto-run limit ${AUTO_RUN_USD}: set CONFIRM=True and re-run this cell (tables already built are skipped)")
    print(f"[{name}] estimated {gb:,.0f} GB ~ ${cost:.2f}")
    job = client.query(sql, job_config=bigquery.QueryJobConfig(
        maximum_bytes_billed=int(gb * 1.25 * 2**30) + 10 * 2**30, use_query_cache=False))
    JOBS.append({"table": name, "job_id": job.job_id, "location": LOCATION,
                 "submitted_at": dt.datetime.now(dt.timezone.utc).isoformat()})
    json.dump(JOBS, open(JOBS_PATH, "w"), indent=2)
    print(f"[{name}] submitted job {job.job_id} (runs on the server; finishes even if Colab disconnects)")
    t0 = time.time()
    job.result()
    billed = (job.total_bytes_billed or 0) / 2**30
    LEDGER["jobs"].append({"table": name, "job_id": job.job_id, "billed_gb": round(billed, 1),
                           "usd": round(usd(billed), 2), "seconds": round(time.time() - t0, 1)})
    LEDGER["usd_total"] = round(LEDGER["usd_total"] + usd(billed), 2)
    json.dump(LEDGER, open(LEDGER_PATH, "w"), indent=2)
    print(f"[{name}] done, {billed:,.0f} GB, {LEDGER['jobs'][-1]['seconds']}s", table_info(name))
    return True

def need(*names):
    missing = [n for n in names if not table_rows(n)]
    assert not missing, f"These tables are not built yet: {missing} (run the earlier cells first)"

def q(sql):
    gb, cost = dry_run(sql)
    assert cost < AUTO_RUN_USD, f"estimated ${cost:.2f}, above the auto-run limit"
    LEDGER["usd_total"] = round(LEDGER["usd_total"] + cost, 4)
    return client.query(sql).to_dataframe()

def upload(name, df):
    df = df.copy()
    for c in df.columns:                       # dates are always stored as DATE
        if pd.api.types.is_datetime64_any_dtype(df[c]):
            df[c] = df[c].dt.date
    job = client.load_table_from_dataframe(
        df, f"{DS}.{name}", job_config=bigquery.LoadJobConfig(write_disposition="WRITE_TRUNCATE"))
    job.result()
    print(f"[{name}] uploaded {len(df):,} rows")

def case_bins(expr, edges, labels, null_label="na"):
    # left-closed, right-open: expr < edges[0] -> labels[0] ...; expr NULL -> null_label (as in 05)
    assert len(labels) == len(edges) + 1
    parts = [f"WHEN {expr} IS NULL THEN '{null_label}'"]
    parts += [f"WHEN {expr} < {e} THEN '{l}'" for e, l in zip(edges, labels[:-1])]
    return "CASE " + " ".join(parts) + f" ELSE '{labels[-1]}' END"

def key_sql(table):
    # the key column of custodial_keys / large_movers may be a BYTES key_id or a hex string
    cols = {f.name: f.field_type for f in client.get_table(f"{DS}.{table}").schema}
    if cols.get("key_id") == "BYTES":
        return f"SELECT DISTINCT key_id FROM `{DS}.{table}`"
    for c in ("key_id_hex", "key_id"):
        if c in cols:
            return f"SELECT DISTINCT FROM_HEX(LOWER({c})) AS key_id FROM `{DS}.{table}`"
    raise RuntimeError(f"no key column found in {table}: {cols}")

for t in ["utxo_life", "address_key", "address_stats", "key_exposure", "key_flags", "key_pubform",
          "exchange_keys", "custodial_keys", "large_movers", "events", "attention_daily",
          "attention_spike_audit", "placebo_days", "placebo_draws_strict", "panel_rs2", "panel_event",
          "holder_snapshot_pc"]:
    client.get_table(f"{DS}.{t}")          # raises if a table is missing
CUST_SQL, LM_SQL = key_sql("custodial_keys"), key_sql("large_movers")
SATS_MIN = int(round(RISK_MIN_BTC * 1e8))
DATA_END = pd.Timestamp(FREEZE_DATE)
print("Required tables found. Spent in this Colab session: $%.2f / $%.0f" % (LEDGER["usd_total"], MAX_USD_TOTAL))

## Step 1 · Local placebo pool and event table

Rules of 03 applied to 2022-02 to 2024-05:

- every Tuesday s; [s − 30, s + 14] contains no event day, event alt_date (Randstorm 10-10, FTX 11-11, …) or detected attention spike (spike window = [start − 30, spike day + 14]);
- maximum general quantum attention within ±3 days of s < 2.

Pools: `pre` (spike window starts at the detected spike day; main), `pre_src` (starts at the hand-verified source day), `pre_sym30` (symmetric ±30 days).

Regression check: applied from 2024-06-04 with the event table of 03 (without `coinbase_migration` and alt_dates) and source-day starts, the same function reproduces `ri_pool` of 03 (35 dates).

In [ ]:
ev_all = q(f"SELECT event, event_set, event_date, alt_date, date_precision FROM `{DS}.events`")
for c in ["event_date", "alt_date"]:
    ev_all[c] = pd.to_datetime(ev_all[c])
att = q(f"SELECT date, attn_pc1 FROM `{DS}.attention_daily`")
att["date"] = pd.to_datetime(att.date)
att = att.set_index("date").sort_index()
audit = q(f"SELECT spike_date, source_date, label FROM `{DS}.attention_spike_audit`")
for c in ["spike_date", "source_date"]:
    audit[c] = pd.to_datetime(audit[c])

E = ev_all.set_index("event")
RS_DATE, FTX_DATE, NOTICE_DATE = E.event_date["randstorm"], E.event_date["ftx"], E.alt_date["randstorm"]
assert NOTICE_DATE == pd.Timestamp("2023-10-10"), f"alt_date of Randstorm in events is {NOTICE_DATE}; expected 2023-10-10"
print("Randstorm public disclosure", RS_DATE.date(), "; individual notification", NOTICE_DATE.date(), "; FTX", FTX_DATE.date())

qmax3 = att.attn_pc1.rolling(7, center=True, min_periods=1).max()     # maximum within +/-3 days (as in 03)

def make_blocked(pre, post, anchor, use_alt=True, drop_events=()):
    # exclude d in [s - pre, s_end + post]; s = event day (and alt_date) or spike (start = anchor column, end = spike day)
    bl = pd.Series(False, index=pd.date_range("2021-01-01", "2027-12-31"))
    for e in ev_all.itertuples():
        if e.event in drop_events:
            continue
        a = e.event_date
        b = (a + pd.offsets.MonthEnd(0)) if e.date_precision == "month" else a
        bl.loc[a - pd.Timedelta(days=pre): b + pd.Timedelta(days=post)] = True
        if use_alt and pd.notna(e.alt_date):
            bl.loc[e.alt_date - pd.Timedelta(days=pre): e.alt_date + pd.Timedelta(days=post)] = True
    for r in audit.itertuples():
        s = getattr(r, anchor)
        bl.loc[s - pd.Timedelta(days=pre): r.spike_date + pd.Timedelta(days=post)] = True
    return bl

def pool_of(lo, hi, pre, post, anchor, **kw):
    bl = make_blocked(pre, post, anchor, **kw)
    grid = pd.date_range(lo, hi, freq="W-TUE")
    return [d for d in grid if not bl.get(d, False) and not (qmax3.get(d, 0) >= SPIKE_SD)]

# check: reproduce the main pool of 03
pdays = q(f"SELECT kind, date FROM `{DS}.placebo_days`")
pdays["date"] = pd.to_datetime(pdays.date)
ri03 = sorted(pdays.loc[pdays.kind == "ri_pool", "date"])
rep = pool_of(PLACEBO_FROM_03, ri03[-1], PRE_EXCL, POST_EXCL, "source_date",
              use_alt=False, drop_events=("coinbase_migration",))
assert rep == ri03, f"the rules do not reproduce the main pool of 03: extra {sorted(set(rep) - set(ri03))}, missing {sorted(set(ri03) - set(rep))}"
rep_spike = pool_of(PLACEBO_FROM_03, ri03[-1], PRE_EXCL, POST_EXCL, "spike_date",
                    use_alt=False, drop_events=("coinbase_migration",))
print(f"Rules reproduce the main pool of 03: {len(rep)} dates match day by day; with spike-day starts: {'identical' if rep_spike == ri03 else 'different'}")

POOLS_PRE = {
    "pre":       pool_of(PRE_FROM, PRE_TO, PRE_EXCL, POST_EXCL, "spike_date"),
    "pre_src":   pool_of(PRE_FROM, PRE_TO, PRE_EXCL, POST_EXCL, "source_date"),
    "pre_sym30": pool_of(PRE_FROM, PRE_TO, SYM_EXCL, SYM_EXCL, "spike_date"),
}
assert set(POOLS_PRE["pre_src"]) <= set(POOLS_PRE["pre"]) and set(POOLS_PRE["pre_sym30"]) <= set(POOLS_PRE["pre"])
for k_, v in POOLS_PRE.items():
    near = {lab: sum(abs((d - e).days) <= 90 for d in v)
            for lab, e in [("FTX", FTX_DATE), ("10-10", NOTICE_DATE), ("11-14", RS_DATE)]}
    print(f"pool {k_:10s} {len(v):3d} dates; by year", pd.Series([d.year for d in v]).value_counts().sort_index().to_dict(),
          "; within +/-90 days:", near)

# original main pool (placebo_draws_strict, 29 dates) - combined pool = local pool + original main pool
dr = q(f"SELECT DISTINCT pool, fake_date FROM `{DS}.placebo_draws_strict`")
dr["fake_date"] = pd.to_datetime(dr.fake_date)
OLD_MAIN = sorted(dr.loc[dr.pool == "main", "fake_date"])
print("Original main pool", len(OLD_MAIN), "dates:", OLD_MAIN[0].date(), "...", OLD_MAIN[-1].date())

pl_rows = [(k_, d, d - pd.Timedelta(days=1), float(att.attn_pc1.get(d, np.nan))) for k_, v in POOLS_PRE.items() for d in v]
placebo_days_rs = pd.DataFrame(pl_rows, columns=["kind", "date", "snap_date", "attn_pc1"])
upload("placebo_days_rs", placebo_days_rs)

ev_rows = [("ftx", FTX_DATE, "positive_control", POST_MAX),
           ("randstorm_notice", NOTICE_DATE, "positive_control", NOTICE_POST_MAX),
           ("randstorm", RS_DATE, "positive_control", POST_MAX)]
ev_rows += [("rp_" + d.strftime("%Y-%m-%d"), d, "placebo_pre", POST_MAX) for d in POOLS_PRE["pre"]]
events_rs = pd.DataFrame(ev_rows, columns=["event", "event_date", "event_set", "post_max"])
events_rs["snap_date"] = events_rs.event_date - pd.Timedelta(days=1)
assert events_rs.event.is_unique
upload("events_rs", events_rs)
pd.concat([placebo_days_rs.assign(source="pre_pools"),
           pd.DataFrame({"kind": "old_main", "date": OLD_MAIN, "source": "placebo_draws_strict"})],
          ignore_index=True).to_csv(f"{OUT_DIR}/est_23_pools.csv", index=False)
print(f"{len(events_rs)} snapshot dates (3 events + {len(POOLS_PRE['pre'])} placebo dates)")

## Step 2 · Randstorm cohort and snapshots, `holder_snapshot_rs`

- `rs_cand`: hash160-family keys (`key_kind = 'k'`) first seen (minimum `first_seen_date` over the key's addresses, as in `key_flags`) between 2012-04-01 and 2015-02-28.
- `holder_snapshot_rs`: balances at the close of t−1 for 62 dates, with the SQL of the positive-control snapshots (02b, 04), except that only addresses of cohort keys are read (all 62 dates in one scan of `utxo_life`), no recent incoming transfer is required, and only group L is kept (key format observable).

Check: on the FTX and Randstorm dates, cohort key counts equal those in `holder_snapshot_pc`.

In [ ]:
SQL_CAND = f'''
CREATE OR REPLACE TABLE `{DS}.rs_cand`
AS
SELECT k.key_id, MIN(s.first_seen_date) AS first_seen_date
FROM `{DS}.address_key` k
JOIN `{DS}.address_stats` s ON s.address = k.address
WHERE k.key_kind = 'k' AND k.key_id IS NOT NULL
GROUP BY k.key_id
HAVING MIN(s.first_seen_date) BETWEEN DATE '{COHORT["wide"][0]}' AND DATE '{COHORT["wide"][1]}'
'''
build("rs_cand", SQL_CAND)

evs = q(f"SELECT event, event_date, snap_date FROM `{DS}.events_rs` ORDER BY event_date")
for c in ["event_date", "snap_date"]:
    evs[c] = pd.to_datetime(evs[c])
MIN_SNAP, MAX_SNAP = evs.snap_date.min(), evs.snap_date.max()
SNAPS_SQL = ",\n    ".join(
    f"STRUCT('{r.event}' AS event, DATE '{r.event_date:%Y-%m-%d}' AS event_date, DATE '{r.snap_date:%Y-%m-%d}' AS snap_date)"
    for r in evs.itertuples())

SQL_SNAP = f'''
CREATE OR REPLACE TABLE `{DS}.holder_snapshot_rs`
CLUSTER BY event
AS
WITH snaps AS (
  SELECT * FROM UNNEST([
    {SNAPS_SQL}
  ])
),
ca AS (
  SELECT k.address, k.key_id
  FROM `{DS}.address_key` k
  JOIN `{DS}.rs_cand` c ON c.key_id = k.key_id
),
u0 AS (   -- filter on the union of all snapshot dates first, then expand to each date
  SELECT u.address, u.type, u.sats, u.created_date, u.spent_date, ca.key_id
  FROM `{DS}.utxo_life` u
  JOIN ca ON ca.address = u.address
  WHERE u.type IN {STD_TYPES}
    AND u.created_date <= DATE '{MAX_SNAP:%Y-%m-%d}'
    AND ( u.spent_date IS NULL OR u.spent_date > DATE '{MIN_SNAP:%Y-%m-%d}'
          OR u.created_date > DATE_SUB(DATE '{MIN_SNAP:%Y-%m-%d}', INTERVAL 365 DAY) )
),
held AS (
  SELECT s.event, s.event_date, s.snap_date, u.key_id, u.address, u.type, u.sats, u.created_date,
         (u.spent_date IS NULL OR u.spent_date > s.snap_date) AS alive
  FROM u0 u
  CROSS JOIN snaps s
  WHERE u.created_date <= s.snap_date
    AND ( u.spent_date IS NULL OR u.spent_date > s.snap_date
          OR u.created_date > DATE_SUB(s.snap_date, INTERVAL 365 DAY) )
),
agg AS (
  SELECT
    a.event, a.event_date, a.snap_date, a.key_id,
    'k'                                            AS key_kind,
    SUM(IF(a.alive, a.sats, 0))                    AS sats,
    COUNTIF(a.alive)                               AS n_utxo,
    COUNT(DISTINCT IF(a.alive, a.address, NULL))   AS n_addr,
    MIN(IF(a.alive, a.created_date, NULL))         AS oldest_utxo_date,
    MAX(IF(a.alive, a.created_date, NULL))         AS newest_utxo_date,
    MAX(a.created_date)                            AS last_in_date
  FROM held a
  GROUP BY a.event, a.event_date, a.snap_date, a.key_id
  HAVING SUM(IF(a.alive, a.sats, 0)) >= {SATS_MIN}
)
SELECT g.*, e.first_exposed_date, e.first_exposed_src, 'L' AS grp, c.first_seen_date
FROM agg g
JOIN `{DS}.rs_cand` c ON c.key_id = g.key_id
JOIN `{DS}.key_exposure` e ON e.key_id = g.key_id
WHERE e.first_exposed_date <= g.snap_date
'''
build("holder_snapshot_rs", SQL_SNAP)

# check: FTX and Randstorm dates identical to the same cohort in holder_snapshot_pc
chk_new = q(f'''
SELECT event, COUNT(*) AS n_keys, SUM(sats) AS sats, SUM(n_utxo) AS n_utxo,
       COUNTIF(last_in_date > DATE_SUB(snap_date, INTERVAL {RISK_LOOKBACK_DAYS} DAY)) AS n_recent
FROM `{DS}.holder_snapshot_rs` WHERE event IN ('ftx', 'randstorm') GROUP BY event ORDER BY event''')
chk_old = q(f'''
SELECT p.event, COUNT(*) AS n_keys, SUM(p.sats) AS sats, SUM(p.n_utxo) AS n_utxo,
       COUNTIF(p.last_in_date > DATE_SUB(p.snap_date, INTERVAL {RISK_LOOKBACK_DAYS} DAY)) AS n_recent
FROM `{DS}.holder_snapshot_pc` p JOIN `{DS}.rs_cand` c ON c.key_id = p.key_id
WHERE p.grp = 'L' AND p.key_kind = 'k' AND p.event IN ('ftx', 'randstorm') GROUP BY p.event ORDER BY p.event''')
display(chk_new.merge(chk_old, on="event", suffixes=("_rs", "_pc")))
assert chk_new.reset_index(drop=True).equals(chk_old.reset_index(drop=True)), "cohort snapshots differ from holder_snapshot_pc"
print("Snapshot check passed: FTX and Randstorm identical to holder_snapshot_pc")

snap_sz = q(f'''
SELECT event, ANY_VALUE(event_date) AS event_date, COUNT(*) AS n_keys,
       COUNTIF(last_in_date > DATE_SUB(snap_date, INTERVAL {RISK_LOOKBACK_DAYS} DAY)) AS n_recent,
       ROUND(SUM(sats) / 1e8) AS btc
FROM `{DS}.holder_snapshot_rs` GROUP BY event ORDER BY event_date''')
display(snap_sz.describe().round(0))

## Step 3 · Public-key format of new keys, `key_pubform_rs`

`key_pubform` of 21 covers only keys in some risk set of 05; cohort keys in the 2022–2023 snapshots outside those risk sets are added here. SQL of 21: public-key push at the end of the P2PKH unlocking script (`41 04…` → U, `21 02/03…` → C), scanning only the monthly partitions with the first spends of these addresses.

In [ ]:
need("holder_snapshot_rs")
MISS = f'''
  SELECT DISTINCT s.key_id
  FROM `{DS}.holder_snapshot_rs` s
  LEFT JOIN `{DS}.key_pubform` p ON p.key_id = s.key_id
  WHERE p.key_id IS NULL
'''
n_miss = int(q(f"SELECT COUNT(*) AS n FROM ({MISS})").n.iloc[0])
n_all = int(q(f"SELECT COUNT(DISTINCT key_id) AS n FROM `{DS}.holder_snapshot_rs`").n.iloc[0])
print(f"{n_all:,} cohort keys in the snapshots, of which {n_miss:,} ({n_miss / max(n_all, 1):.2%}) are not in key_pubform")

if n_miss == 0:
    build("key_pubform_rs", f"CREATE OR REPLACE TABLE `{DS}.key_pubform_rs` AS SELECT * FROM `{DS}.key_pubform` WHERE FALSE")
else:
    months = q(f'''
    WITH m AS ({MISS})
    SELECT DISTINCT DATE_TRUNC(s.first_spend_date, MONTH) AS mo
    FROM `{DS}.address_key` k
    JOIN m ON m.key_id = k.key_id
    JOIN `{DS}.address_stats` s ON s.address = k.address
    WHERE STARTS_WITH(k.address, '1') AND s.first_spend_date IS NOT NULL
      AND s.first_spend_date <= DATE '{FREEZE_DATE}'
    ''')
    MONTHS = sorted(pd.to_datetime(months.mo))
    RANGES, a = [], None
    for i_, mth in enumerate(MONTHS):          # merge consecutive months into ranges; constant ranges prune partitions (as in 21)
        if a is None: a = mth
        nxt = MONTHS[i_ + 1] if i_ + 1 < len(MONTHS) else None
        if nxt is None or nxt != mth + pd.offsets.MonthBegin(1):
            RANGES.append((a, mth + pd.offsets.MonthBegin(1))); a = None
    TS_FILTER = " OR ".join(f"(i.block_timestamp >= TIMESTAMP '{x:%Y-%m-%d}' AND i.block_timestamp < TIMESTAMP '{y:%Y-%m-%d}')"
                            for x, y in RANGES) or "FALSE"
    print(f"First spends fall in {len(MONTHS)} months, {len(RANGES)} consecutive ranges")
    SQL_PF = f'''
CREATE OR REPLACE TABLE `{DS}.key_pubform_rs` AS
WITH m AS ({MISS}),
ad AS (
  SELECT k.address, k.key_id
  FROM `{DS}.address_key` k
  JOIN m ON m.key_id = k.key_id
  JOIN `{DS}.address_stats` s ON s.address = k.address
  WHERE STARTS_WITH(k.address, '1') AND s.first_spend_date IS NOT NULL
),
i AS (
  SELECT a AS address, i.script_hex
  FROM `{INPUTS}` i, UNNEST(i.addresses) AS a
  WHERE ({TS_FILTER})
    AND i.type = 'pubkeyhash'
),
f AS (
  SELECT ad.key_id,
    CASE WHEN LENGTH(i.script_hex) >= 132 AND SUBSTR(i.script_hex, LENGTH(i.script_hex) - 131, 4) = '4104' THEN 'U'
         WHEN LENGTH(i.script_hex) >= 68 AND SUBSTR(i.script_hex, LENGTH(i.script_hex) - 67, 2) = '21'
              AND SUBSTR(i.script_hex, LENGTH(i.script_hex) - 65, 2) IN ('02', '03') THEN 'C'
         ELSE 'X' END AS form
  FROM i JOIN ad ON ad.address = i.address
)
SELECT key_id, COUNTIF(form = 'U') AS n_u, COUNTIF(form = 'C') AS n_c, COUNTIF(form = 'X') AS n_x,
  CASE WHEN COUNTIF(form = 'U') > 0 AND COUNTIF(form = 'C') = 0 THEN 'U'
       WHEN COUNTIF(form = 'C') > 0 AND COUNTIF(form = 'U') = 0 THEN 'C'
       WHEN COUNTIF(form = 'U') > 0 THEN 'mixed' ELSE 'unknown' END AS pubform
FROM f
GROUP BY key_id
'''
    build("key_pubform_rs", SQL_PF)

PF_SQL = f"SELECT key_id, pubform FROM `{DS}.key_pubform` UNION ALL SELECT key_id, pubform FROM `{DS}.key_pubform_rs`"
cov = q(f'''
SELECT e.event_set, IFNULL(pf.pubform, 'not_found') AS pubform, COUNT(*) AS n
FROM `{DS}.holder_snapshot_rs` s
JOIN `{DS}.events_rs` e ON e.event = s.event
LEFT JOIN ({PF_SQL}) pf ON pf.key_id = s.key_id
GROUP BY 1, 2 ORDER BY 1, 3 DESC''')
display(cov.pivot(index="pubform", columns="event_set", values="n").fillna(0).astype(int))
dup = int(q(f"SELECT COUNT(*) AS n FROM (SELECT key_id FROM ({PF_SQL}) GROUP BY key_id HAVING COUNT(*) > 1)").n.iloc[0])
assert dup == 0, "key_pubform and key_pubform_rs share keys"

## Step 4 · Exclusion flags and outflow destinations (rules of 04)

- `key_flags_rs`: keys in `key_flags` keep its `is_service`; new keys get the same rule (exchange keys ∪ any address with > 1,000 lifetime receipts ∪ `custodial_keys`). Check: for keys in `key_flags`, the recomputed rule agrees with `key_flags.is_service`.
- `flow_days_rs`: relative days 0…+30 per date (0…+65 for 10-10), truncated at the freeze date.
- `flow_tx_inputs_rs` → `flow_tx_outputs_rs`, `dest_key_info_rs` → `flow_key_tx_rs`: SQL of 04 with the risk set replaced by the cohort keys.
- Check: identities of 04 (spent = change + fee + y3; y3 = y2 + y0 + y1 loose + other; shares sum to ≤ 1).

In [ ]:
need("holder_snapshot_rs", "key_pubform_rs")
SQL_FLAGS = f'''
CREATE OR REPLACE TABLE `{DS}.key_flags_rs`
AS
WITH ks AS (SELECT DISTINCT key_id FROM `{DS}.holder_snapshot_rs`),
st AS (
  SELECT k.key_id, MAX(s.n_received) AS max_addr_n_received
  FROM `{DS}.address_key` k
  JOIN ks ON ks.key_id = k.key_id
  JOIN `{DS}.address_stats` s ON s.address = k.address
  GROUP BY k.key_id
),
ck AS ({CUST_SQL})
SELECT st.key_id, st.max_addr_n_received,
       ek.key_id IS NOT NULL AS is_exchange,
       ck.key_id IS NOT NULL AS is_custodial,
       kf.key_id IS NOT NULL AS in_key_flags,
       kf.is_service         AS kf_is_service,
       (ek.key_id IS NOT NULL OR st.max_addr_n_received > {SERVICE_MAX_RECV} OR ck.key_id IS NOT NULL) AS rule_is_service,
       IF(kf.key_id IS NOT NULL, kf.is_service,
          ek.key_id IS NOT NULL OR st.max_addr_n_received > {SERVICE_MAX_RECV} OR ck.key_id IS NOT NULL) AS is_service
FROM st
LEFT JOIN `{DS}.exchange_keys` ek ON ek.key_id = st.key_id
LEFT JOIN ck ON ck.key_id = st.key_id
LEFT JOIN `{DS}.key_flags` kf ON kf.key_id = st.key_id
'''
build("key_flags_rs", SQL_FLAGS)
fl = q(f'''
SELECT COUNT(*) AS n_keys, COUNTIF(in_key_flags) AS in_key_flags, COUNTIF(is_service) AS n_service,
       COUNTIF(in_key_flags AND kf_is_service != rule_is_service) AS rule_mismatch
FROM `{DS}.key_flags_rs`''')
display(fl)
assert fl.rule_mismatch.iloc[0] == 0, "recomputed exclusion rule differs from key_flags.is_service"
assert fl.n_keys.iloc[0] == n_all

# dates to tag
evr = q(f"SELECT event, event_set, event_date, post_max FROM `{DS}.events_rs`")
evr["event_date"] = pd.to_datetime(evr.event_date)
fd = [(r.event, r.event_set, r.event_date, k, r.event_date + pd.Timedelta(days=k))
      for r in evr.itertuples() for k in range(0, int(r.post_max) + 1)
      if r.event_date + pd.Timedelta(days=k) <= DATA_END]
flow_days_rs = pd.DataFrame(fd, columns=["event", "event_set", "event_date", "rel_day", "date"])
upload("flow_days_rs", flow_days_rs)
FLOW_MIN, FLOW_MAX = flow_days_rs.date.min(), flow_days_rs.date.max()
print(f"{len(evr)} dates, {len(flow_days_rs):,} rows, {flow_days_rs.date.nunique()} distinct days: {FLOW_MIN.date()} ... {FLOW_MAX.date()}")

# inputs: inputs_sql of 04 with cohort keys as the risk set
SQL_IN = f'''
CREATE OR REPLACE TABLE `{DS}.flow_tx_inputs_rs`
PARTITION BY spent_date
AS
WITH d AS (SELECT DISTINCT date FROM `{DS}.flow_days_rs`),
rk AS (SELECT key_id FROM `{DS}.key_flags_rs` WHERE NOT is_service),
inp AS (
  SELECT u.spending_txid, u.spent_date, u.address, u.type, u.sats, u.created_date,
         k.key_id
  FROM `{DS}.utxo_life` u
  JOIN d ON u.spent_date = d.date
  LEFT JOIN `{DS}.address_key` k ON k.address = u.address
)
SELECT i.*, rk.key_id IS NOT NULL AS is_risk_key
FROM inp i
LEFT JOIN rk ON rk.key_id = i.key_id
             AND i.type IN {STD_TYPES}
WHERE TRUE
QUALIFY LOGICAL_OR(rk.key_id IS NOT NULL) OVER (PARTITION BY i.spending_txid)
'''
build("flow_tx_inputs_rs", SQL_IN)

SQL_OUT = f'''
CREATE OR REPLACE TABLE `{DS}.flow_tx_outputs_rs`
PARTITION BY created_date
AS
WITH tx AS (SELECT DISTINCT spending_txid AS txid, spent_date FROM `{DS}.flow_tx_inputs_rs`)
SELECT u.txid, u.vout, u.created_date, u.created_height, u.sats, u.type, u.address,
       IF(u.spent_date <= DATE '{DATA_END:%Y-%m-%d}', u.spent_date, NULL) AS out_spent_date,
       k.key_id                    AS dest_key_id,
       k.key_kind                  AS dest_key_kind,
       s.first_seen_height         AS addr_first_seen_height
FROM `{DS}.utxo_life` u
JOIN tx ON tx.txid = u.txid AND tx.spent_date = u.created_date
LEFT JOIN `{DS}.address_key`   k ON k.address = u.address
LEFT JOIN `{DS}.address_stats` s ON s.address = u.address
WHERE u.created_date BETWEEN DATE '{FLOW_MIN:%Y-%m-%d}' AND DATE '{FLOW_MAX:%Y-%m-%d}'
'''
build("flow_tx_outputs_rs", SQL_OUT)

SQL_DK = f'''
CREATE OR REPLACE TABLE `{DS}.dest_key_info_rs`
AS
WITH dk AS (SELECT DISTINCT dest_key_id AS key_id FROM `{DS}.flow_tx_outputs_rs` WHERE dest_key_id IS NOT NULL),
fs AS (
  SELECT k.key_id, MIN(s.first_seen_height) AS key_first_seen_height
  FROM `{DS}.address_key` k
  JOIN dk ON dk.key_id = k.key_id
  JOIN `{DS}.address_stats` s ON s.address = k.address
  GROUP BY k.key_id
)
SELECT fs.key_id, fs.key_first_seen_height, e.first_exposed_date
FROM fs
LEFT JOIN `{DS}.key_exposure` e ON e.key_id = fs.key_id
'''
build("dest_key_info_rs", SQL_DK)

SQL_KTX = f'''
CREATE OR REPLACE TABLE `{DS}.flow_key_tx_rs`
PARTITION BY date
AS
WITH tx_in AS (
  SELECT spending_txid AS txid, ANY_VALUE(spent_date) AS date, SUM(sats) AS in_sats, COUNT(*) AS n_in,
         ARRAY_AGG(DISTINCT key_id IGNORE NULLS) AS in_keys
  FROM `{DS}.flow_tx_inputs_rs` GROUP BY txid
),
key_in AS (
  SELECT spending_txid AS txid, key_id, SUM(sats) AS key_in_sats, COUNT(*) AS n_utxo_in,
         MIN(created_date) AS oldest_in_created, MAX(created_date) AS newest_in_created
  FROM `{DS}.flow_tx_inputs_rs` WHERE is_risk_key GROUP BY txid, key_id
),
o AS (
  SELECT o.txid, o.sats,
    IFNULL(o.dest_key_id IN UNNEST(t.in_keys), FALSE)                               AS is_change,
    ex.key_id IS NOT NULL                                                            AS to_ex,
    IFNULL(ex.src = 'por', FALSE)                                                    AS to_ex_por,
    (o.type IN ('pubkey', 'multisig', 'witness_v1_taproot')
       OR IFNULL(dk.first_exposed_date <= o.created_date, FALSE))                    AS exposed,
    o.dest_key_id IS NOT NULL                                                        AS keyed,
    IFNULL(dk.key_first_seen_height = o.created_height, FALSE)                       AS new_key,
    IFNULL(o.addr_first_seen_height = o.created_height, FALSE)
      AND o.type IN ('pubkeyhash', 'witness_v0_keyhash', 'scripthash', 'witness_v0_scripthash')
      AND (o.out_spent_date IS NULL
           OR o.out_spent_date > DATE_ADD(o.created_date, INTERVAL {DORMANT_DAYS} DAY))  AS strict_ok,
    DATE_ADD(o.created_date, INTERVAL {DORMANT_DAYS} DAY) > DATE '{DATA_END:%Y-%m-%d}' AS censored
  FROM `{DS}.flow_tx_outputs_rs` o
  JOIN tx_in t ON t.txid = o.txid
  LEFT JOIN `{DS}.dest_key_info_rs` dk ON dk.key_id = o.dest_key_id
  LEFT JOIN `{DS}.exchange_keys` ex ON ex.key_id = o.dest_key_id
),
c AS (
  SELECT *,
    NOT is_change                                              AS nc,
    NOT is_change AND to_ex                                    AS c_y2,
    NOT is_change AND NOT to_ex AND exposed                    AS c_y0,
    NOT is_change AND NOT to_ex AND NOT exposed AND keyed      AS c_y1l,
    NOT is_change AND NOT to_ex AND NOT exposed AND NOT keyed  AS c_oth
  FROM o
),
tx_out AS (
  SELECT txid,
    SUM(sats)                                      AS out_sats,
    COUNT(*)                                       AS n_out,
    SUM(IF(is_change, sats, 0))                    AS change_sats,
    SUM(IF(nc, sats, 0))                           AS y3,
    SUM(IF(c_y2, sats, 0))                         AS y2,
    SUM(IF(nc AND to_ex_por, sats, 0))             AS y2_por,
    SUM(IF(c_y0, sats, 0))                         AS y0,
    SUM(IF(c_y1l, sats, 0))                        AS y1_loose,
    SUM(IF(c_y1l AND new_key, sats, 0))            AS y1_mid,
    SUM(IF(c_y1l AND strict_ok, sats, 0))          AS y1_strict,
    SUM(IF(c_y1l AND strict_ok AND censored, sats, 0)) AS y1_strict_censored,
    SUM(IF(c_oth, sats, 0))                        AS other
  FROM c GROUP BY txid
)
SELECT
  ki.key_id, t.date, ki.txid,
  ki.key_in_sats, ki.n_utxo_in, ki.oldest_in_created, ki.newest_in_created,
  t.in_sats, t.n_in, ARRAY_LENGTH(t.in_keys) AS n_in_keys, IFNULL(o.n_out, 0) AS n_out,
  SAFE_DIVIDE(ki.key_in_sats, t.in_sats)                          AS share,
  SAFE_DIVIDE(ki.key_in_sats, t.in_sats) * IFNULL(o.change_sats, 0)          AS change_sats,
  SAFE_DIVIDE(ki.key_in_sats, t.in_sats) * (t.in_sats - IFNULL(o.out_sats, 0)) AS fee_sats,
  SAFE_DIVIDE(ki.key_in_sats, t.in_sats) * IFNULL(o.y3, 0)                   AS y3,
  SAFE_DIVIDE(ki.key_in_sats, t.in_sats) * IFNULL(o.y2, 0)                   AS y2,
  SAFE_DIVIDE(ki.key_in_sats, t.in_sats) * IFNULL(o.y2_por, 0)               AS y2_por,
  SAFE_DIVIDE(ki.key_in_sats, t.in_sats) * IFNULL(o.y0, 0)                   AS y0,
  SAFE_DIVIDE(ki.key_in_sats, t.in_sats) * IFNULL(o.y1_loose, 0)             AS y1_loose,
  SAFE_DIVIDE(ki.key_in_sats, t.in_sats) * IFNULL(o.y1_mid, 0)               AS y1_mid,
  SAFE_DIVIDE(ki.key_in_sats, t.in_sats) * IFNULL(o.y1_strict, 0)            AS y1_strict,
  SAFE_DIVIDE(ki.key_in_sats, t.in_sats) * IFNULL(o.y1_strict_censored, 0)   AS y1_strict_censored,
  SAFE_DIVIDE(ki.key_in_sats, t.in_sats) * IFNULL(o.other, 0)                AS other
FROM key_in ki
JOIN tx_in t ON t.txid = ki.txid
LEFT JOIN tx_out o ON o.txid = ki.txid
'''
build("flow_key_tx_rs", SQL_KTX)

chk = q(f'''
SELECT
  ROUND(SUM(key_in_sats)/1e8, 4)                                         AS spent_btc,
  ROUND(SUM(change_sats + fee_sats + y3 - key_in_sats)/1e8, 6)           AS gap_spent_identity,
  ROUND(SUM(y2 + y0 + y1_loose + other - y3)/1e8, 6)                     AS gap_y3_identity,
  COUNTIF(y1_strict > y1_loose + 1) AS bad_strict, COUNTIF(y1_mid > y1_loose + 1) AS bad_mid,
  COUNTIF(fee_sats < -1) AS neg_fee
FROM `{DS}.flow_key_tx_rs`''')
share_chk = q(f'''
SELECT COUNTIF(s > 1 + 1e-9) AS tx_share_gt_1
FROM (SELECT txid, SUM(share) AS s FROM `{DS}.flow_key_tx_rs` GROUP BY txid)''')
display(chk)
c_ = chk.iloc[0]
assert abs(c_.gap_spent_identity) < 1e-3 * max(c_.spent_btc, 1) and abs(c_.gap_y3_identity) < 1e-3 * max(c_.spent_btc, 1)
assert c_.bad_strict == 0 and c_.bad_mid == 0 and c_.neg_fee == 0 and share_chk.tx_share_gt_1.iloc[0] == 0
print("Destination identity checks passed")

## Step 5 · Panels (as in 05 and 21)

- `panel_keys_rs`: cohort keys × dates; `is_service` dropped; four CEM bins (balance, coin age, last incoming transfer, number of UTXOs); arm = public-key format U / C; `is_large_mover` = large-transfer keys of 04c ∪ keys with daily y3 ≥ 1,000 BTC in the new windows.
- `panel_key_flows_rs`: cumulative shares (capped at 1) per key × date × relative day, including the held-coins version.
- `panel_hazard_rs`: relative day of first de-risking (cumulative y1 strict ≥ 50%).
- `panel_rs2_ext`: cells and columns of `panel_rs2` (21).

Regression check against `panel_rs2` on the FTX and Randstorm dates (date × window × arm × CEM cell, relative days 7 and 30): equal key counts; sums of cumulative shares, BTC-weighted sums and first de-riskings within 1e-6.

In [ ]:
need("flow_key_tx_rs", "key_flags_rs", "key_pubform_rs", "events_rs", "flow_days_rs")
LM_MIN_SATS = int(LM_MIN_BTC * 1e8)
SQL_PK = f'''
CREATE OR REPLACE TABLE `{DS}.panel_keys_rs`
CLUSTER BY event
AS
WITH pf AS ({PF_SQL}),
kday AS (SELECT key_id, date, SUM(y3) AS y3 FROM `{DS}.flow_key_tx_rs` GROUP BY key_id, date),
lm AS (
  SELECT key_id FROM ({LM_SQL})
  UNION DISTINCT
  SELECT DISTINCT key_id FROM kday WHERE y3 >= {LM_MIN_SATS}
),
b AS (
  SELECT s.event, s.event_date, s.snap_date, s.key_id, s.sats, s.n_utxo, s.oldest_utxo_date, s.last_in_date,
         s.first_seen_date, e.event_set, e.post_max,
         LEAST(e.post_max, DATE_DIFF(DATE '{FREEZE_DATE}', s.event_date, DAY))           AS censor_day,
         s.last_in_date > DATE_SUB(s.snap_date, INTERVAL {RISK_LOOKBACK_DAYS} DAY)        AS recent_in,
         IFNULL(pf.pubform, 'not_found')                                                  AS arm,
         lm.key_id IS NOT NULL                                                            AS is_large_mover,
         s.first_seen_date BETWEEN DATE '{COHORT["main"][0]}' AND DATE '{COHORT["main"][1]}' AS in_main,
         s.first_seen_date BETWEEN DATE '{COHORT["wide"][0]}' AND DATE '{COHORT["wide"][1]}' AS in_wide
  FROM `{DS}.holder_snapshot_rs` s
  JOIN `{DS}.events_rs` e ON e.event = s.event
  JOIN `{DS}.key_flags_rs` f ON f.key_id = s.key_id
  LEFT JOIN pf ON pf.key_id = s.key_id
  LEFT JOIN lm ON lm.key_id = s.key_id
  WHERE NOT f.is_service
)
SELECT b.*,
  {case_bins("sats", [int(e * 1e8) for e in BAL_EDGES_BTC], BAL_LABELS)}               AS bal_bin,
  {case_bins("DATE_DIFF(snap_date, oldest_utxo_date, DAY)", AGE_EDGES_D, AGE_LABELS)}  AS age_bin,
  {case_bins("DATE_DIFF(snap_date, last_in_date, DAY)", REC_EDGES_D, REC_LABELS)}      AS rec_bin,
  {case_bins("n_utxo", NUTXO_EDGES, NUTXO_LABELS)}                                     AS nutxo_bin
FROM b
'''
build("panel_keys_rs", SQL_PK)

sum_cols = ",\n    ".join([f"SUM(t.{o}) AS {o}" for o in OUTS if o not in HELD] +
                          [f"SUM(IF(t.newest_in_created <= pk.snap_date, t.{src}, 0)) AS {o}" for o, src in HELD.items()])
cum_cols = ",\n    ".join(f"SUM({o}) OVER w AS cy_{o}" for o in OUTS)
vc_cols  = ",\n  ".join(f"{o}, LEAST(1.0, cy_{o} / bal) AS vc_{o}" for o in OUTS)
SQL_PKF = f'''
CREATE OR REPLACE TABLE `{DS}.panel_key_flows_rs`
CLUSTER BY event
AS
WITH t AS (
  SELECT fd.event, fd.rel_day, f.key_id, f.newest_in_created,
         {", ".join(f"f.{o}" for o in OUTS if o not in HELD)}
  FROM `{DS}.flow_key_tx_rs` f
  JOIN `{DS}.flow_days_rs` fd ON fd.date = f.date
),
a AS (
  SELECT t.event, t.key_id, t.rel_day, ANY_VALUE(pk.sats) AS bal,
    {sum_cols}
  FROM t
  JOIN `{DS}.panel_keys_rs` pk ON pk.event = t.event AND pk.key_id = t.key_id
  GROUP BY t.event, t.key_id, t.rel_day
),
c AS (
  SELECT a.*,
    {cum_cols}
  FROM a
  WINDOW w AS (PARTITION BY event, key_id ORDER BY rel_day ROWS BETWEEN UNBOUNDED PRECEDING AND CURRENT ROW)
)
SELECT event, key_id, rel_day, bal,
  {vc_cols}
FROM c
'''
build("panel_key_flows_rs", SQL_PKF)

build("panel_hazard_rs", f'''
CREATE OR REPLACE TABLE `{DS}.panel_hazard_rs`
CLUSTER BY event
AS
SELECT event, key_id, MIN(IF(vc_y1_strict >= 0.5, rel_day, NULL)) AS t_y1s_50
FROM `{DS}.panel_key_flows_rs`
GROUP BY event, key_id
''')

v_cols = ",\n    ".join(f"MAX(IF(f.rel_day <= r, f.vc_{o}, 0)) AS v_{o}" for o in OUTS)
agg = ",\n  ".join(f"SUM(IFNULL(f.v_{o}, 0)) AS c_{o}, SUM(POW(IFNULL(f.v_{o}, 0), 2)) AS c2_{o}, "
                   f"SUM(kd.sats * IFNULL(f.v_{o}, 0)) / 1e8 AS cb_{o}" for o in OUTS)
SQL_RS2X = f'''
CREATE OR REPLACE TABLE `{DS}.panel_rs2_ext`
CLUSTER BY event
AS
WITH k AS (
  SELECT event, key_id, sats, bal_bin, age_bin, rec_bin, nutxo_bin, is_large_mover, arm, in_main, in_wide, censor_day
  FROM `{DS}.panel_keys_rs`
  WHERE arm IN ('U', 'C')
),
kw AS (
  SELECT k.*, 'main' AS w FROM k WHERE in_main
  UNION ALL
  SELECT k.*, 'wide' AS w FROM k WHERE in_wide
),
f AS (
  SELECT f.event, f.key_id, r,
    {v_cols}
  FROM `{DS}.panel_key_flows_rs` f, UNNEST(GENERATE_ARRAY(0, {NOTICE_POST_MAX})) AS r
  GROUP BY f.event, f.key_id, r
),
h AS (SELECT event, key_id, t_y1s_50 FROM `{DS}.panel_hazard_rs`),
kd AS (
  SELECT kw.*, r FROM kw, UNNEST(GENERATE_ARRAY(0, kw.censor_day)) AS r
)
SELECT kd.event, kd.w AS win, kd.arm, CONCAT(kd.bal_bin, '|', kd.age_bin, '|', kd.rec_bin, '|', kd.nutxo_bin) AS stratum,
  kd.bal_bin, kd.rec_bin, kd.is_large_mover, kd.r AS rel_day,
  COUNT(*) AS n, SUM(kd.sats) / 1e8 AS bal,
  {agg},
  COUNTIF(h.t_y1s_50 <= kd.r) AS c_h50
FROM kd
LEFT JOIN f ON f.event = kd.event AND f.key_id = kd.key_id AND f.r = kd.r
LEFT JOIN h ON h.event = kd.event AND h.key_id = kd.key_id
GROUP BY kd.event, win, kd.arm, stratum, kd.bal_bin, kd.rec_bin, kd.is_large_mover, rel_day
'''
build("panel_rs2_ext", SQL_RS2X)

# regression check against panel_rs2 of 21 (large-mover flag is key-level: merged over cells)
CMP_COLS = ["n", "bal", "c_y1_strict", "cb_y1_strict", "c_y1_strict_held", "c_y3", "c_y0", "c_h50"]
cmp_sql = lambda t: f'''
  SELECT event, win, arm, stratum, rel_day, {", ".join(f"SUM({c}) AS {c}" for c in CMP_COLS)}
  FROM `{DS}.{t}` WHERE event IN ('ftx', 'randstorm') AND rel_day IN (7, 30)
  GROUP BY event, win, arm, stratum, rel_day'''
new_c, old_c = q(cmp_sql("panel_rs2_ext")), q(cmp_sql("panel_rs2"))
mm = new_c.merge(old_c, on=["event", "win", "arm", "stratum", "rel_day"], how="outer", suffixes=("_new", "_old"), indicator=True)
only = mm[mm._merge != "both"]
diffs = {c: float(np.nanmax(np.abs(mm[f"{c}_new"].astype(float) - mm[f"{c}_old"].astype(float)))) for c in CMP_COLS}
tot = mm.groupby(["event", "win", "arm", "rel_day"])[[f"n_new", "n_old", "c_y1_strict_new", "c_y1_strict_old"]].sum()
display(tot)
print("Cells present on one side only:", len(only), "; max absolute difference by column:", {k: f"{v:.2e}" for k, v in diffs.items()})
assert len(only) == 0 and diffs["n"] == 0, "cells or key counts differ from panel_rs2 of 21"
assert all(v < 1e-6 for v in diffs.values()), "cumulative shares differ from panel_rs2 of 21"
print("Panel check passed: FTX and Randstorm match 21 cell by cell")

sz = q(f'''
SELECT e.event_set, x.win, x.arm, COUNT(DISTINCT x.event) AS n_dates,
       AVG(x.n_day) AS mean_keys, MIN(x.n_day) AS min_keys,
       AVG(x.n_rs2) AS mean_keys_rs2, MIN(x.n_rs2) AS min_keys_rs2
FROM (SELECT event, win, arm, SUM(n) AS n_day, SUM(IF(rec_bin != '365d+', n, 0)) AS n_rs2
      FROM `{DS}.panel_rs2_ext` WHERE rel_day = 0 GROUP BY 1, 2, 3) x
JOIN `{DS}.events_rs` e ON e.event = x.event
GROUP BY 1, 2, 3 ORDER BY 1, 2, 3''')
display(sz.round(0))

## Step 6 · Estimation

Estimator and inference of 20 and 21: U − C θ per date and relative day; `RS2` (incoming transfer within 12 months; main) and `RS2_all`.

Pools: `pre` (main), `pre_src`, `pre_sym30`, `all` (local + the 29 dates of the original main pool), `old_main` (original main pool; reproduction of 21 only). `RS2_all` uses the local pools only, since the original pool snapshots require recent incoming transfers.

Regression check: `old_main` with h = 90 reproduces the Randstorm and FTX results of 21 (θ within 1e-9, β within 1e-6).

In [ ]:
need("panel_rs2_ext")
ext = q(f"SELECT * FROM `{DS}.panel_rs2_ext`")
old = q(f"SELECT * FROM `{DS}.panel_rs2` WHERE event NOT IN ('ftx', 'randstorm')")
assert set(ext.columns) == set(old.columns), f"columns differ: {set(ext.columns) ^ set(old.columns)}"
rs2 = pd.concat([ext[old.columns], old], ignore_index=True)

ev_new = q(f"SELECT event, event_set, event_date FROM `{DS}.events_rs`")
ev_old = q(f'''SELECT event, ANY_VALUE(event_set) AS event_set, ANY_VALUE(event_date) AS event_date
               FROM `{DS}.panel_event` WHERE rel_day = 0 GROUP BY event''')
ev_old = ev_old[~ev_old.event.isin(ev_new.event)]
ev = pd.concat([ev_new, ev_old], ignore_index=True)
ev["event_date"] = pd.to_datetime(ev.event_date)
ev = ev[ev.event.isin(rs2.event.unique())].sort_values("event_date").reset_index(drop=True)
print(f"{len(ev)} dates:", ev.event_set.value_counts().to_dict())

def theta(df):
    # df: cell rows of one event and rel_day (including arm). Returns key-weighted / BTC-weighted U - C for each outcome (as in 21)
    g = df.groupby(["stratum", "arm"]).sum(numeric_only=True).unstack("arm").fillna(0)
    if ("n", "U") not in g or ("n", "C") not in g:
        return {}
    g = g[(g[("n", "U")] > 0) & (g[("n", "C")] > 0)]
    nU, nC, bU, bC = g[("n", "U")], g[("n", "C")], g[("bal", "U")], g[("bal", "C")]
    out = {"n_U_matched": nU.sum(), "n_C_matched": nC.sum()}
    for o in [f"c_{x}" for x in OUTS] + ["c_h50"]:
        cU, cC = g[(o, "U")], g[(o, "C")]
        out[f"{o}|key|diff"] = (cU.sum() - (nU / nC * cC).sum()) / nU.sum()
        out[f"{o}|key|T"] = cU.sum() / nU.sum()
        out[f"{o}|key|C"] = (nU / nC * cC).sum() / nU.sum()
        if o != "c_h50":
            ob = o.replace("c_", "cb_")
            out[f"{o}|btc|diff"] = (g[(ob, "U")].sum() - (bU / bC * g[(ob, "C")]).sum()) / bU.sum()
            out[f"{o}|btc|T"] = g[(ob, "U")].sum() / bU.sum()
    return out

recs, sizes = [], []
for (win, e, r), d in rs2.groupby(["win", "event", "rel_day"]):
    full = r in REPORT_DAYS                       # subgroups and no_lm only on reported days; paths need only all/all
    for contrast, dd in [("RS2", d[d.rec_bin != "365d+"]), ("RS2_all", d)]:
        for variant, ddd in [("all", dd), ("no_lm", dd[~dd.is_large_mover])][: 2 if full else 1]:
            sgs = [("all", ddd)] + ([(b, ddd[ddd.bal_bin == b]) for b in sorted(ddd.bal_bin.unique())] if full else [])
            for sg, dx in sgs:
                t = theta(dx)
                if r == 0 and variant == "all" and sg == "all" and t:
                    sizes.append((win, contrast, e, t["n_U_matched"], t["n_C_matched"]))
                for k_, v in t.items():
                    if "|" in k_:
                        o, w, stat = k_.split("|")
                        recs.append((win, contrast, variant, sg, o, w, stat, r, e, v))
L = pd.DataFrame(recs, columns=["win", "contrast", "variant", "subgroup", "outcome", "weight", "stat", "rel_day", "event", "value"])
SER = ["win", "contrast", "variant", "subgroup", "outcome", "weight", "stat", "rel_day"]
TH = L.pivot_table(index=SER, columns="event", values="value", aggfunc="first").reindex(columns=ev.event.tolist())
SIZES = pd.DataFrame(sizes, columns=["win", "contrast", "event", "n_U", "n_C"])
print("Series:", len(TH))
display(SIZES[SIZES.event.isin(["ftx", "randstorm_notice", "randstorm"])])

# pools
pd_rs = q(f"SELECT kind, date FROM `{DS}.placebo_days_rs`")
pd_rs["date"] = pd.to_datetime(pd_rs.date)
dr = q(f"SELECT DISTINCT pool, fake_date FROM `{DS}.placebo_draws_strict`")
dr["fake_date"] = pd.to_datetime(dr.fake_date)
pl_old = ev[ev.event_set == "placebo"].set_index("event_date").event
OLD_MAIN = sorted(dr.loc[dr.pool == "main", "fake_date"])
assert all(d in pl_old.index for d in OLD_MAIN), "dates of the original main pool not found in panel_rs2"
POOL_EVENTS = {k_: ["rp_" + d.strftime("%Y-%m-%d") for d in sorted(pd_rs.loc[pd_rs.kind == k_, "date"])]
               for k_ in ["pre", "pre_src", "pre_sym30"]}
POOL_EVENTS["old_main"] = [pl_old[d] for d in OLD_MAIN]
POOL_EVENTS["all"] = POOL_EVENTS["pre"] + POOL_EVENTS["old_main"]
for k_, v in POOL_EVENTS.items():
    miss = [x for x in v if x not in ev.event.values]
    assert not miss, f"dates of pool {k_} not found in the panel: {miss[:5]}"
    print(f"pool {k_:10s}: {len(v)} dates")

T0 = pd.Timestamp("2020-01-01")
tday = ((ev.event_date - T0).dt.days).to_numpy(float)
EV_IDX = {e: i for i, e in enumerate(ev.event)}

def smoother(pool, h, mask=None):
    # returns S (events x pool); dates > 60 days before the pool use the pool mean
    idx = np.array([EV_IDX[e] for e in POOL_EVENTS[pool]])
    tp = tday[idx]
    ok = np.ones(len(idx), bool) if mask is None else mask
    lo = tp[ok].min() - 60
    S = np.zeros((len(ev), len(idx)))
    for i in range(len(ev)):
        w = np.exp(-0.5 * ((tp - tday[i]) / h) ** 2) * ok
        self_ = idx == i
        w[self_] = 0
        if tday[i] < lo:
            w = ok.astype(float); w[self_] = 0
            S[i] = w / w.sum(); continue
        X = np.column_stack([np.ones_like(tp), tp - tday[i]])
        A = X.T @ (X * w[:, None])
        S[i] = np.linalg.solve(A, (X * w[:, None]).T)[0]
    return S

def effects(Th, pool, h):
    idx = [EV_IDX[e] for e in POOL_EVENTS[pool]]
    Y = Th.to_numpy(float)
    P = Y[:, idx]
    B = np.full_like(Y, np.nan)
    nanmask = np.isnan(P)
    keys = pd.Series([m.tobytes() for m in nanmask])
    for key, rows_ in keys.groupby(keys).groups.items():
        rows_ = np.array(list(rows_))
        ok = ~nanmask[rows_[0]]
        if ok.sum() < 5:
            continue
        S = smoother(pool, h, ok)
        B[rows_] = Y[rows_] - np.nan_to_num(P[rows_]) @ S.T
    R = B[:, idx]
    return pd.DataFrame(B, index=Th.index, columns=Th.columns), R

def ri_p(beta, R):
    b = np.abs(np.asarray(beta, float))[:, :, None]
    r = np.abs(R)[:, None, :]
    cnt = np.sum(r >= b - 1e-15, axis=2)
    n = np.sum(~np.isnan(R), axis=1)[:, None]
    p = (1 + cnt) / (1 + n)
    p[np.isnan(np.asarray(beta, float))] = np.nan
    return p

def ri_ci(beta, R, a=0.10):
    lo_q, hi_q = np.nanquantile(R, 1 - a / 2, axis=1), np.nanquantile(R, a / 2, axis=1)
    return beta - lo_q[:, None], beta - hi_q[:, None]

REPORT = ["ftx", "randstorm_notice", "randstorm"]
diff = TH.xs("diff", level="stat")
lev_T = TH.xs("T", level="stat").reindex(diff.index)
lev_C = TH.xs("C", level="stat").reindex(diff.index)
out, KEEP = [], {}
for pool in ["pre", "pre_src", "pre_sym30", "all", "old_main"]:
    for h in BANDWIDTHS:
        B, R = effects(diff, pool, h)
        p = ri_p(B[REPORT].to_numpy(), R)
        lo, hi = ri_ci(B[REPORT].to_numpy(), R)
        sd = np.nanstd(R, axis=1, ddof=1)
        npool = np.sum(~np.isnan(R), axis=1)
        for j, e in enumerate(REPORT):
            out.append(pd.DataFrame({"event": e, "pool": pool, "h": h, "theta": diff[e].values, "beta": B[e].values,
                                     "p_ri": p[:, j], "ci90_lo": lo[:, j], "ci90_hi": hi[:, j],
                                     "level_T": lev_T[e].values, "level_C": lev_C[e].values,
                                     "z_placebo_sd": B[e].values / sd, "n_pool": npool}, index=diff.index).reset_index())
        KEEP[(pool, h)] = (B, R)
eff = pd.concat(out, ignore_index=True)
eff = eff[~((eff.contrast == "RS2_all") & eff.pool.isin(["all", "old_main"]))]       # different definitions; not reported
eff = eff.join(ev.set_index("event")[["event_set", "event_date"]], on="event")
eff_rep = eff[eff.rel_day.isin(REPORT_DAYS)]
eff_rep.to_csv(f"{OUT_DIR}/est_23_pc.csv", index=False)

# regression check against 21 (old_main, h = 90)
s21 = eff_rep[(eff_rep.pool == "old_main") & (eff_rep.h == 90) & (eff_rep.win == "main") & (eff_rep.contrast == "RS2")
              & (eff_rep.variant == "all") & (eff_rep.subgroup == "all") & (eff_rep.weight == "key") & (eff_rep.rel_day == 30)]
REP_CHECK = []
for (e, o), (th_ref, b_ref) in REF_21.items():
    r_ = s21[(s21.event == e) & (s21.outcome == o)].iloc[0]
    REP_CHECK.append({"event": e, "outcome": o, "theta": r_.theta, "theta_21": th_ref, "beta": r_.beta, "beta_21": b_ref})
REP_CHECK = pd.DataFrame(REP_CHECK)
display(REP_CHECK)
assert (np.abs(REP_CHECK.theta - REP_CHECK.theta_21) < 1e-9).all() and (np.abs(REP_CHECK.beta - REP_CHECK.beta_21) < 1e-6).all(), \
    "the results of 21 were not reproduced"
print("Reproduction of 21 passed")

MAINSPEC = (eff_rep.win == "main") & (eff_rep.variant == "all") & (eff_rep.subgroup == "all") & (eff_rep.weight == "key")
show = eff_rep[MAINSPEC & (eff_rep.h == H_MAIN) & eff_rep.outcome.isin(["c_y1_strict", "c_h50", "c_y1_strict_held", "c_y0"])]
display(show.pivot_table(index=["event", "contrast", "outcome", "rel_day"], columns="pool",
                         values=["beta", "p_ri", "z_placebo_sd"]).round(4))

In [ ]:
# calendar-time figure (U - C, RS2, main window, y1 strict, key-weighted, +30) and the 10-10 path
KEY30 = ("main", "RS2", "all", "all", "c_y1_strict", "key", 30)
th30 = diff.loc[KEY30]
B90, R90 = KEEP[("all", H_MAIN)]
cal = ev[["event", "event_set", "event_date"]].copy()
cal["theta"] = cal.event.map(th30)
cal["baseline_all_h90"] = cal.event.map(th30 - B90.loc[KEY30])       # leave-one-out baseline for dates in the pool
for k_ in ["pre", "pre_src", "pre_sym30", "old_main"]:
    cal[f"in_{k_}"] = cal.event.isin(POOL_EVENTS[k_])
cal = cal[cal.theta.notna()]

def curve(pool, h, ts):
    # value of the local linear fit to the pool theta at calendar dates ts (for plotting)
    idx = np.array([EV_IDX[e] for e in POOL_EVENTS[pool]])
    tp, yp = tday[idx], th30.reindex(POOL_EVENTS[pool]).to_numpy(float)
    ok = ~np.isnan(yp)
    tp, yp = tp[ok], yp[ok]
    out_ = []
    for t in ts:
        w = np.exp(-0.5 * ((tp - t) / h) ** 2)
        X = np.column_stack([np.ones_like(tp), tp - t])
        out_.append(np.linalg.solve(X.T @ (X * w[:, None]), (X * w[:, None]).T @ yp)[0])
    return np.array(out_)

grid = pd.date_range(ev.event_date.min(), ev.event_date.max(), freq="7D")
fit = curve("all", H_MAIN, ((grid - T0).days).to_numpy(float))
rk = R90[diff.index.get_loc(KEY30)]
q05, q95 = np.nanquantile(rk, 0.05), np.nanquantile(rk, 0.95)
pd.concat([cal.assign(kind="date"),
           pd.DataFrame({"kind": "curve", "event_date": grid, "baseline_all_h90": fit,
                         "band_lo": fit + q05, "band_hi": fit + q95})], ignore_index=True
          ).to_csv(f"{OUT_DIR}/est_23_calendar_uc.csv", index=False)

fig, ax = plt.subplots(figsize=(11, 4.6))
pp = cal[cal.in_pre | cal.in_old_main]
ax.fill_between(grid, (fit + q05) * 100, (fit + q95) * 100, color="0.85", lw=0, label="placebo 5–95% band")
ax.plot(grid, fit * 100, color="0.35", lw=1.2, label="local-linear baseline (h = 90 d)")
ax.scatter(pp.event_date, pp.theta * 100, s=14, color="0.45", zorder=3, label="placebo dates")
for e, lab, col in [("ftx", "FTX", "#0072B2"), ("randstorm_notice", "Randstorm notice (10 Oct 2023)", "#E69F00"),
                    ("randstorm", "Randstorm disclosure (14 Nov 2023)", "#D55E00")]:
    r_ = cal[cal.event == e]
    if len(r_):
        ax.scatter(r_.event_date, r_.theta * 100, s=60, color=col, zorder=4, label=lab, marker="D")
ax.axhline(0, color="k", lw=0.5)
ax.set_ylabel("Uncompressed − compressed, 30-day share (pp)")
ax.set_title("Randstorm cohort: event dates against same-period placebo dates", fontsize=10)
ax.legend(fontsize=7, ncol=3, loc="upper center", bbox_to_anchor=(0.5, -0.12), frameon=False)
for s in ["top", "right"]: ax.spines[s].set_visible(False)
plt.tight_layout(); plt.savefig(f"{OUT_DIR}/fig_23_calendar_uc.png", dpi=200); plt.show()

# 10-10 notification event: same snapshot followed to +65 (11-14 = relative day 35), descriptive
NP = diff.xs(("main", "RS2", "all", "all", "c_y1_strict", "key"), level=["win", "contrast", "variant", "subgroup", "outcome", "weight"])
LT = lev_T.xs(("main", "RS2", "all", "all", "c_y1_strict", "key"), level=["win", "contrast", "variant", "subgroup", "outcome", "weight"])
LC = lev_C.xs(("main", "RS2", "all", "all", "c_y1_strict", "key"), level=["win", "contrast", "variant", "subgroup", "outcome", "weight"])
EVD = ev.set_index("event").event_date
gap = int((EVD["randstorm"] - EVD["randstorm_notice"]).days)
npath = pd.DataFrame({"rel_day": NP.index, "theta_notice": NP["randstorm_notice"].values,
                      "U_level": LT["randstorm_notice"].values, "C_level": LC["randstorm_notice"].values})
npath["theta_disclosure_shifted"] = npath.rel_day.map(lambda r: NP["randstorm"].get(r - gap, np.nan))
npath.to_csv(f"{OUT_DIR}/est_23_notice_path.csv", index=False)
fig, ax = plt.subplots(figsize=(8, 3.6))
ax.plot(npath.rel_day, npath.theta_notice * 100, color="#E69F00", lw=1.6, label="snapshot 9 Oct 2023: U − C cumulative")
ax.plot(npath.rel_day, npath.theta_disclosure_shifted * 100, color="#D55E00", lw=1.2, ls="--",
        label="snapshot 13 Nov 2023 (survivors): U − C cumulative")
ax.axvline(gap, color="0.4", ls=":", lw=1); ax.text(gap + 0.5, ax.get_ylim()[1] * 0.95, "public disclosure", fontsize=7, va="top")
ax.axhline(0, color="k", lw=0.5)
ax.set_xlabel("days since 10 Oct 2023"); ax.set_ylabel("pp of pre-event balance")
ax.legend(fontsize=7, frameon=False, loc="upper left")
for s in ["top", "right"]: ax.spines[s].set_visible(False)
plt.tight_layout(); plt.savefig(f"{OUT_DIR}/fig_23_notice_path.png", dpi=200); plt.show()

In [ ]:
main_rows = eff_rep[MAINSPEC & (eff_rep.h == H_MAIN) & eff_rep.outcome.isin(["c_y1_strict", "c_h50"])]
report = {
    "generated_at": dt.datetime.now(dt.timezone.utc).isoformat(),
    "ledger_usd_this_session": LEDGER["usd_total"],
    "pools": {k_: len(v) for k_, v in POOL_EVENTS.items()},
    "pool_dates": {k_: v for k_, v in POOL_EVENTS.items()},
    "reproduce_21": REP_CHECK.to_dict(orient="records"),
    "sizes_r0": SIZES[SIZES.event.isin(REPORT)].to_dict(orient="records"),
    "main_spec": main_rows[["event", "contrast", "outcome", "rel_day", "pool", "theta", "beta", "p_ri",
                            "ci90_lo", "ci90_hi", "level_T", "level_C", "z_placebo_sd", "n_pool"]].to_dict(orient="records"),
    "tables": {t: table_info(t) for t in ["rs_cand", "holder_snapshot_rs", "key_pubform_rs", "key_flags_rs",
                                          "flow_tx_inputs_rs", "flow_key_tx_rs", "panel_keys_rs", "panel_rs2_ext"]},
}
with open(f"{OUT_DIR}/build_report_23.json", "w", encoding="utf-8") as f:
    json.dump(report, f, ensure_ascii=False, indent=2, default=str)
with zipfile.ZipFile("out23.zip", "w", zipfile.ZIP_DEFLATED) as z:
    for fn in sorted(os.listdir(OUT_DIR)):
        z.write(os.path.join(OUT_DIR, fn), arcname=f"out23/{fn}")
print("Packed out23.zip:", sorted(os.listdir(OUT_DIR)))
try:
    from google.colab import files
    files.download("out23.zip")
except Exception:
    print("Not in Colab: use out23.zip directly")

In [ ]:
for rec in JOBS[-5:]:
    job = client.get_job(rec["job_id"], location=rec.get("location", LOCATION))
    print(rec["table"], job.state, "errors:", job.errors)
    if job.state != "DONE":
        job.result(); print("  done")
for t in ["placebo_days_rs", "events_rs", "rs_cand", "holder_snapshot_rs", "key_pubform_rs", "key_flags_rs", "flow_days_rs",
          "flow_tx_inputs_rs", "flow_tx_outputs_rs", "dest_key_info_rs", "flow_key_tx_rs",
          "panel_keys_rs", "panel_key_flows_rs", "panel_hazard_rs", "panel_rs2_ext"]:
    n = table_rows(t)
    print(f"{t:22s}", f"{n:,} rows" if n is not None else "not built yet")